In [40]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests

In [41]:
# models
model_gemini='gemini-3-flash-preview'
model_gpt = 'gpt-4o-mini'
model_llama = 'gemma4'


In [3]:
#creating gemini client

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


gemini=OpenAI(api_key=google_api_key,base_url=GEMINI_BASE_URL)
print("your gemini client is ready to use")

API key found and looks good so far!
your gemini client is ready to use


In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [5]:
#creating Ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
print("your ollama client is ready to use")

your ollama client is ready to use


In [67]:
relevant_link_system_prompt="""   
You are a marketing assistant specialized in selecting relevant links from a list of URLs.

Your task:
- Analyze the provided URLs.
- Select only the links that are useful for a marketing brochure.



Ignore:
- Login / Signup
- Privacy Policy
- Terms & Conditions
- Admin / Dashboard
- Duplicate or irrelevant links

Output Rules:
- Return ONLY valid JSON (no explanations, no extra text).
- Use EXACTLY this format:

{
    "links": [
        {"type": "<page type>", "url": "<full url>"},
        {"type": "<page type>", "url": "<full url>"}
    ]
}

Type Rules:
- Use clear lowercase values such as:
  "about page", "product page", "pricing page", "contact page",
  "blog", "case studies", "testimonials", "careers page"


- No duplicates

Additional instruction:
- Infer the page type from the URL if not explicitly clear.

"""

In [68]:

def get_relevant_links_user_prompt(company_links):
     relevant_links_user_prompt=f""""Company Name: 

Here is a list of URLs from the company website:

{company_links}

Select the most relevant links for a marketing brochure based on their usefulness to customers.

Return ONLY JSON as specified.

"""
     return relevant_links_user_prompt


In [69]:

def get_relevant_links(url):
    #fetching website links
    company_links=fetch_website_links(url)
   
    #sending request
    response = gemini.chat.completions.create(model=model_gemini, messages=[{"role": "system", "content":relevant_link_system_prompt},{"role":"user","content": get_relevant_links_user_prompt(company_links)}],response_format={"type":"json_object"})
    
    #loading links in json object
    links = json.loads(response.choices[0].message.content)
    return links
   

In [ ]:
get_relevant_links(url="https://huggingface.co")

{'links': [{'type': 'product page', 'url': '/models'},
  {'type': 'product page', 'url': '/datasets'},
  {'type': 'product page', 'url': '/spaces'},
  {'type': 'enterprise page', 'url': '/enterprise'},
  {'type': 'pricing page', 'url': '/pricing'},
  {'type': 'blog', 'url': '/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learning center', 'url': '/learn'},
  {'type': 'product page', 'url': '/chat'},
  {'type': 'brand resources', 'url': '/brand'}]}

In [66]:
def fetch_relevant_links_content(url):
    contents = fetch_website_contents(url)
    relevant_links = get_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n##Relevant  Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
fetch_relevant_links_content(url)

In [ ]:
brochure_system_prompt="""You are an expert marketing copywriter and brand strategist.

Your task is to generate a high-quality, persuasive marketing brochure based on provided website content.

Guidelines:
- Write in a professional, engaging, and concise tone.
- Focus on value proposition, benefits, and differentiation.
- Avoid copying text verbatim; rewrite and enhance it.
- Structure the brochure clearly with sections.
- Use simple, compelling language suitable for customers (not technical unless needed).
- Highlight key services, products, and strengths.
- Remove irrelevant, repetitive, or noisy content.
- If information is missing, infer reasonably but do not hallucinate specific facts.

Output Format (Markdown, no code blocks):
- Company Name (as title)
- Tagline (short and catchy)
- About Us
- Products / Services
- Key Features / Advantages
- Why Choose Us
- Call to Action (CTA)

Important:
- Do NOT include raw URLs or navigation elements unless useful.
- Do NOT mention “based on the provided content”.
- Ensure the brochure is polished and ready for real-world use."""

In [ ]:
def get_brochure_user_prompt():
    

In [48]:
# request llm model to answer, with streaming
def get_relevant_links(url,model):
    #fetching website links
    company_links=fetch_website_links(url)
    get_relevant_links_user_prompt(company_links)

    #sending request
    if(model=='gemini'):

       stream = gemini.chat.completions.create(model=model_gemini, messages=[{"role": "system", "content":relevant_link_system_prompt},{"role":"user","content": get_relevant_links_user_prompt(company_links)}],stream=True)
    else:
        stream = ollama.chat.completions.create(model="gemma4", messages=[{"role": "system", "content":relevant_link_system_prompt},{"role":"user","content": get_relevant_links_user_prompt(company_links)}],stream=True)

    
    #displaying chunks
    response=""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
           response += chunk.choices[0].delta.content or ''
           update_display(Markdown(response), display_id=display_handle.display_id) 